In [1]:
import sys

model_path = '/Users/rdobrien/skysurf/Zodiacal_Light_Model_Python'
if model_path not in sys.path:
    sys.path.append(model_path)

import time
import healpy as hp
import numpy as np
import zodisurf

In [2]:
lats = np.arange(-90, 90, 10) # Plot a range of latitudes
wave = 2.2
day_main = 150.0

days = [day_main]*len(lats) # All input arrays (except wavelength) must be the same length
lons = [0.0]*len(lats)

# Calculate intensities for all latitudes at once
intensities = zodisurf.get_zmod(wave, phase_type = 'kelsall', day = days, lon = lons, lat = lats)

In [3]:
# from zodisurf.zkernel_numba import warmup
# warmup('skysurf')           # compile now

In [4]:
# Set map resolution
nside = 128
npix = hp.nside2npix(nside)

# Retrieve pixel coordinates in ecliptic longitude and latitude (degrees)
pixels = np.arange(npix)
lon, lat = hp.pix2ang(nside, pixels, lonlat=True)

# Start the timer
start_time = time.time()

# Calculate zodiacal light values across all pixels
# (Note: If get_zmod is not natively vectorized, wrap with np.vectorize or a loop)
zodi_map = zodisurf.get_zmod(
    lambda_=1.0, phase_type='skysurf', day=[200]*len(pixels), lon=lon, lat=lat, L2=True, engine = 'numba'
)

# Stop the timer
end_time = time.time()
elapsed_time = end_time - start_time

print(
    f'Successfully populated nside={nside} map ({npix} pixels) in {elapsed_time:.4f} seconds.'
)

Successfully populated nside=128 map (196608 pixels) in 7.3777 seconds.


In [5]:
rng = np.random.default_rng(7)
n = 3000
lon = rng.uniform(0, 360, n)
lat = np.degrees(np.arcsin(rng.uniform(-1, 1, n)))
day = rng.uniform(1, 12000, n)

for phase, lam in (('skysurf', 1.0), ('kelsall', 2.2)):
    for L2 in (True, False):
        a = zodisurf.get_zmod(lam, phase, day, lon, lat, L2=L2, engine='numpy')
        b = zodisurf.get_zmod(lam, phase, day, lon, lat, L2=L2, engine='numba')
        d = np.abs(a - b)
        rel = np.max(d / np.maximum(np.abs(a), 1e-30))
        print(f"{phase:8s} L2={L2!s:5s}  max|diff|={d.max():.3g}  "
              f"max rel={rel:.3g}  exact={int((a == b).sum())}/{n}")

skysurf  L2=True   max|diff|=0  max rel=0  exact=3000/3000
skysurf  L2=False  max|diff|=0  max rel=0  exact=3000/3000
kelsall  L2=True   max|diff|=0  max rel=0  exact=3000/3000
kelsall  L2=False  max|diff|=0  max rel=0  exact=3000/3000


In [9]:
# diag_cluster.py  -- run with:  python diag_cluster.py
import os, sys, time, platform
import numpy as np, healpy as hp
import numba

print("host:", platform.node(), "| python", sys.version.split()[0])
print("numpy", np.__version__, "| numba", numba.__version__)
print("cpu_count:", os.cpu_count(),
      "| affinity:", len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else "n/a")
print("numba threads:", numba.config.NUMBA_NUM_THREADS, "| threading layer env:",
      os.environ.get("NUMBA_THREADING_LAYER"))
print("NUMBA_* env:", {k: v for k, v in os.environ.items() if k.startswith("NUMBA")})
print("load avg:", os.getloadavg())
try:
    import llvmlite; print("llvmlite", llvmlite.__version__)
    from llvmlite import binding as llvm
    print("LLVM cpu:", llvm.get_host_cpu_name())
except Exception as e:
    print("llvmlite info failed:", e)
print("SVML in use:", getattr(numba.config, "USING_SVML", "n/a"))

nside = 128
npix = hp.nside2npix(nside)
lon, lat = hp.pix2ang(nside, np.arange(npix), lonlat=True)
sirr = float(np.ravel(zodisurf.solar_sp(2.0))[0])
FIT = dict(fitting_hong=[0.366, -0.3, -0.87, 0.0496, 0.03, 0.0003], fitting_albedo=0.2,
           fitting_em_smooth=1.0, fitting_em_band=1.0, fitting_em_ring=1.0)

zodisurf.get_zmod(1.0, 'skysurf', [200.0], [10.0], [10.0], L2=True, engine='numpy')   # warm-up
# for label, args, kw in [
#     ("A skysurf (your benchmark)", ('skysurf', 1.0, 200.0), {}),
#     ("E fitting (the mock's call)", ('fitting', 2.0, 11284.0), dict(solar_irr=sirr, **FIT)),
# ]:
#     t0 = time.perf_counter()
#     zodisurf.get_zmod(lambda_=args[1], phase_type=args[0], day=np.full(npix, args[2]),
#                       lon=lon, lat=lat, L2=True, engine='numba', n_jobs=1, **kw)
#     print(f"{label:<30s} {time.perf_counter() - t0:7.2f} s")

for label, args, kw in [
    ("A skysurf", ('skysurf', 1.0, 200.0), {}),
    ("E fitting", ('fitting', 2.0, 11284.0), dict(solar_irr=sirr, **FIT)),
]:
    for rep in range(3):
        t0 = time.perf_counter()
        zodisurf.get_zmod(lambda_=args[1], phase_type=args[0], day=np.full(npix, args[2]),
                          lon=lon, lat=lat, L2=True, engine='numba', n_jobs=1, **kw)
        print(f"{label:<12s} rep{rep}: {time.perf_counter() - t0:7.2f} s")

host: gs66-ankylosaur | python 3.11.18
numpy 2.1.3 | numba 0.68.0
cpu_count: 12 | affinity: n/a
numba threads: 12 | threading layer env: None
NUMBA_* env: {}
load avg: (1.67578125, 1.63134765625, 1.71533203125)
llvmlite 0.50.0
LLVM cpu: apple-m2
SVML in use: False
A skysurf    rep0:    7.07 s
A skysurf    rep1:    7.18 s
A skysurf    rep2:    7.19 s
E fitting    rep0:    7.24 s
E fitting    rep1:    7.43 s
E fitting    rep2:    7.26 s
